In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional

from src.utils import *
from src.analysis import *
from src.model import FineTunedModel



In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FineTunedModel(num_classes=10).to(device)
model.load_state_dict(torch.load('weights/finetune_weights.pth', map_location=device))
target_layer = model.feature_extractor[5]
D = torch.load('weights/hals_nnd_D.pth')


In [ ]:
subset_paths = load_subset(subset_root="data/subset")
analysis_result = run_complete_analysis(
    model=model,
    target_layer=target_layer,
    D=torch.load('weights/hals_nnd_D.pth'),
    subset_paths=subset_paths,
    k=5,
    lam=1e-2,
    batch_size=10
)

In [ ]:
def visualize_atom_overlap_matrix(overlap_matrix: np.ndarray, class_names: List[str]):
    """Visualize the atom overlap matrix as a heatmap."""
    plt.figure(figsize=(5, 4))
    
    # Mask diagonal
    mask = np.eye(10, dtype=bool)
    overlap_masked = np.ma.masked_where(mask, overlap_matrix)
    
    plt.imshow(overlap_masked, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(label='Jaccard Similarity')
    
    # Set ticks
    plt.xticks(range(10), class_names, rotation=45, ha='right', fontsize=6)
    plt.yticks(range(10), class_names, fontsize=6)
    
    # Add values
    for i in range(10):
        for j in range(10):
            if i != j and overlap_matrix[i, j] > 0:
                plt.text(j, i, f'{overlap_matrix[i, j]:.2f}', 
                        ha='center', va='center', fontsize=4)
    
    plt.title('Top-5 Characteristic Atoms Overlap Between Classes', 
              fontsize=7, fontweight='bold')
    plt.xlabel('Class')
    plt.ylabel('Class')
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/atom_overlap_matrix.png', dpi=150, bbox_inches='tight')
    print("\nSaved atom overlap visualization to: figures/atom_overlap_matrix.png")
    plt.show()




In [ ]:
def analyze_class_characteristic_atoms(results: List[dict], n_atoms: int, 
                                       top_k_atoms: int = 10, threshold: float = 0.1,
                                       top_channels: int = 10):
    """
    Analyze which atoms are characteristic for each class.
    
    Args:
        results: List of analysis results with 'label', 'path', and 'analysis' keys
        n_atoms: Total number of atoms in dictionary
        top_k_atoms: Number of top atoms to report per class
        threshold: Minimum average weight to consider an atom as characteristic
        top_channels: Number of top influential channels to average (default: 10)
        
    Returns:
        Dictionary with class-specific atom statistics
    """
    # Extract class names from paths in results
    # Path format: 'data/subset/class_name/image.jpeg'
    class_name_set = set()
    label_to_class_name = {}
    
    for r in results:
        path = r['path']
        label = r['label']
        # Extract class name from path (second to last component)
        class_name = path.split('/')[-2]
        class_name_set.add(class_name)
        label_to_class_name[label] = class_name
    
    # Sort class names by their label indices
    sorted_labels = sorted(label_to_class_name.keys())
    class_names = [label_to_class_name[label] for label in sorted_labels]
    n_classes = len(class_names)
    
    print(f"\nDetected {n_classes} classes: {class_names}")
    
    # Aggregate sparse codes by class
    class_codes = {i: [] for i in range(n_classes)}
    
    for r in results:
        label = r['label']
        sparse_codes = r['analysis']['sparse_codes']  # (n_channels, n_atoms)
        weights = r['analysis']['weights']  # (n_channels,)
        
        # Get top-K most influential channels
        top_channel_indices = np.argsort(weights)[-top_channels:]
        
        # Average across only top-K channels for this image
        top_codes = sparse_codes[top_channel_indices]  # (top_channels, n_atoms)
        avg_code = np.mean(np.abs(top_codes), axis=0)  # (n_atoms,)
        class_codes[label].append(avg_code)
    
    # Compute statistics per class
    class_stats = {}
    
    for class_idx in range(n_classes):
        if len(class_codes[class_idx]) == 0:
            continue
        
        codes = np.array(class_codes[class_idx])  # (n_images, n_atoms)
        
        # Compute mean and std for each atom across images in this class
        mean_activation = np.mean(codes, axis=0)  # (n_atoms,)
        std_activation = np.std(codes, axis=0)
        
        # Compute consistency: how often each atom is used (non-zero)
        usage_rate = np.mean(codes > 1e-6, axis=0)  # (n_atoms,)
        
        class_stats[class_idx] = {
            'mean_activation': mean_activation,
            'std_activation': std_activation,
            'usage_rate': usage_rate,
            'n_images': len(codes)
        }
    
    # Find characteristic atoms for each class
    print("\n" + "="*80)
    print("CLASS-CHARACTERISTIC ATOMS ANALYSIS")
    print("="*80)
    
    class_characteristic_atoms = {}
    
    for class_idx in range(n_classes):
        if class_idx not in class_stats:
            continue
        
        stats = class_stats[class_idx]
        mean_act = stats['mean_activation']
        usage = stats['usage_rate']
        
        # Score: combination of activation strength and usage frequency
        # High score = strong activation AND frequently used in this class
        scores = mean_act * usage
        
        # Get top-k atoms
        top_indices = np.argsort(scores)[-top_k_atoms:][::-1]
        
        # Filter by threshold
        characteristic = []
        for atom_idx in top_indices:
            if scores[atom_idx] >= threshold:
                characteristic.append({
                    'atom_idx': int(atom_idx),
                    'score': float(scores[atom_idx]),
                    'mean_activation': float(mean_act[atom_idx]),
                    'usage_rate': float(usage[atom_idx])
                })
        
        class_characteristic_atoms[class_idx] = characteristic
        
        # Print results
        print(f"\n{class_names[class_idx].upper()} (Class {class_idx}):")
        print(f"  Images analyzed: {stats['n_images']}")
        print(f"  (Using top-{top_channels} influential channels per image)")
        print(f"  Characteristic atoms (score ≥ {threshold}):")
        
        if len(characteristic) > 0:
            for rank, atom_info in enumerate(characteristic, 1):
                print(f"    #{rank:2d}  Atom {atom_info['atom_idx']:3d}  |  "
                      f"Score: {atom_info['score']:6.3f}  |  "
                      f"Avg Activation: {atom_info['mean_activation']:6.3f}  |  "
                      f"Usage: {atom_info['usage_rate']*100:5.1f}%")
        else:
            print(f"    No atoms exceed threshold {threshold}")
    
    # Cross-class analysis: find atoms shared vs unique
    print("\n" + "="*80)
    print("CROSS-CLASS ATOM ANALYSIS")
    print("="*80)
    
    # Get top-5 atoms for each class for overlap analysis
    top_atoms_per_class = {}
    for class_idx, atoms in class_characteristic_atoms.items():
        if len(atoms) > 0:
            top_atoms_per_class[class_idx] = [a['atom_idx'] for a in atoms[:5]]
    
    # Find shared atoms (appear in multiple classes)
    atom_class_map = {}
    for class_idx, atom_list in top_atoms_per_class.items():
        for atom_idx in atom_list:
            if atom_idx not in atom_class_map:
                atom_class_map[atom_idx] = []
            atom_class_map[atom_idx].append(class_idx)
    
    shared_atoms = {atom: classes for atom, classes in atom_class_map.items() 
                    if len(classes) > 1}
    unique_atoms = {atom: classes[0] for atom, classes in atom_class_map.items() 
                    if len(classes) == 1}
    
    print(f"\nShared atoms (used by multiple classes): {len(shared_atoms)}")
    if len(shared_atoms) > 0:
        for atom_idx, class_list in sorted(shared_atoms.items(), 
                                          key=lambda x: len(x[1]), reverse=True)[:10]:
            class_names_list = [class_names[c] for c in class_list]
            print(f"  Atom {atom_idx:3d}: {', '.join(class_names_list)} ({len(class_list)} classes)")
    
    print(f"\nClass-unique atoms (top-5 only in one class): {len(unique_atoms)}")
    for class_idx in range(n_classes):
        unique_for_class = [atom for atom, cls in unique_atoms.items() if cls == class_idx]
        if len(unique_for_class) > 0:
            print(f"  {class_names[class_idx]:12s}: {unique_for_class}")
    
    # Compute inter-class atom overlap
    print("\n" + "="*80)
    print("PAIRWISE CLASS ATOM OVERLAP (Jaccard Similarity)")
    print("="*80)
    
    overlap_matrix = np.zeros((n_classes, n_classes))
    for i in range(n_classes):
        for j in range(n_classes):
            if i in top_atoms_per_class and j in top_atoms_per_class:
                atoms_i = set(top_atoms_per_class[i])
                atoms_j = set(top_atoms_per_class[j])
                overlap_matrix[i, j] = jaccard_sim(list(atoms_i), list(atoms_j))
    
    # Print matrix
    print("\n      ", end="")
    for i in range(n_classes):
        print(f"{class_names[i][:4]:>5s}", end="")
    print()
    
    for i in range(n_classes):
        print(f"{class_names[i][:4]:>5s}:", end="")
        for j in range(n_classes):
            if overlap_matrix[i, j] > 0:
                print(f"{overlap_matrix[i, j]:5.2f}", end="")
            else:
                print(f"  -  ", end="")
        print()
    
    # Visualize overlap matrix
    visualize_atom_overlap_matrix(overlap_matrix, class_names)
    
    return {
        'class_stats': class_stats,
        'characteristic_atoms': class_characteristic_atoms,
        'shared_atoms': shared_atoms,
        'unique_atoms': unique_atoms,
        'overlap_matrix': overlap_matrix,
        'class_names': class_names  # Include class names in return value
    }

In [ ]:
atom_analysis = analyze_class_characteristic_atoms(
        results=analysis_result['results'],
        n_atoms=len(D),
        top_k_atoms=10,
        threshold=0.1
    )

In [ ]:
plt.figure(figsize=(2, 2))
plt.imshow(D[107].cpu().numpy(), cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
atom_analysis.keys()

In [ ]:
def visualize_unique_atoms_grid(atom_analysis: dict, D: torch.Tensor, 
                                 figsize: tuple = (6, 6), cmap: str = 'gray'):
    """
    Visualize the first unique atom for each class in a 3x3 grid.
    
    Args:
        atom_analysis: Output from analyze_class_characteristic_atoms containing 'unique_atoms'
        D: Dictionary tensor of atoms (n_atoms, H, W)
        figsize: Figure size
        cmap: Colormap for visualization
    """
    unique_atoms = atom_analysis['unique_atoms']
    class_names = atom_analysis['class_names']
    
    # Group unique atoms by class
    class_unique_atoms = {}
    for atom_idx, class_idx in unique_atoms.items():
        if class_idx not in class_unique_atoms:
            class_unique_atoms[class_idx] = []
        class_unique_atoms[class_idx].append(atom_idx)
    
    # Get classes with unique atoms (sorted by class index)
    classes_with_unique = sorted(class_unique_atoms.keys())
    n_classes_with_unique = len(classes_with_unique)
    
    print(f"\nVisualizing {n_classes_with_unique} classes with unique atoms")
    
    # Create 3x3 grid
    fig, axes = plt.subplots(3, 3, figsize=figsize)
    axes = axes.flatten()
    
    for idx, class_idx in enumerate(classes_with_unique):
        # if idx >= 9:  # Only show first 9
        #     break
            
        ax = axes[idx]
        
        # Get first unique atom for this class
        first_atom_idx = class_unique_atoms[class_idx][0]
        atom = D[first_atom_idx].cpu().numpy()
        
        # Display atom
        im = ax.imshow(atom, cmap=cmap)
        ax.set_title(f"{class_names[class_idx]}\nAtom {first_atom_idx}", 
                     fontsize=6, fontweight='bold')
        ax.axis('off')
        
        # Add colorbar
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    # Hide unused subplots
    for idx in range(n_classes_with_unique, 9):
        axes[idx].axis('off')
    
    plt.suptitle('Class-Unique Atoms (First Unique Atom per Class)', 
                 fontsize=10, fontweight='bold', y=0.98)
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/unique_atoms_grid.png', dpi=150, bbox_inches='tight')
    print("Saved visualization to: figures/unique_atoms_grid.png")
    plt.show()


# Usage example:
visualize_unique_atoms_grid(atom_analysis, D)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T
import os

# ============================================================================
# 1. VISUALIZE ATOM INPUT SPACE (Optimized Input)
# ============================================================================

def denormalize_tensor(tensor):
    """Chuyển tensor (đã normalize theo ImageNet) về ảnh RGB chuẩn để hiển thị."""
    mean = torch.tensor([0.485, 0.456, 0.406], device=tensor.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=tensor.device).view(3, 1, 1)
    tensor = tensor * std + mean
    img_np = torch.clamp(tensor, 0, 1).permute(1, 2, 0).cpu().numpy()
    return img_np

def visualize_atom_input_space(model, atom, layer_index, steps=200, lr=0.1, device='cuda'):
    """Tạo ra ảnh đầu vào kích hoạt mạnh nhất một Atom cụ thể (Global Optimization)."""
    model.eval()
    random_img = torch.randn(1, 3, 224, 224, device=device) * 0.01
    random_img.requires_grad_(True)
    optimizer = optim.Adam([random_img], lr=lr)
    atom_tensor = torch.tensor(atom, device=device).float()
    
    target_activations = []
    def hook_fn(module, input, output):
        target_activations.append(output)
    
    handle = model.feature_extractor[layer_index].register_forward_hook(hook_fn)
    
    try:
        for i in range(steps):
            optimizer.zero_grad()
            target_activations.clear()
            _ = model(random_img)
            
            act = target_activations[0].squeeze(0) 
            spatial_activation = torch.mean(act, dim=0)
            
            # Loss function: Maximize correlation with Atom pattern
            loss = -torch.sum(spatial_activation * atom_tensor)
            
            loss.backward()
            optimizer.step()
            
            # Regularization (optional but recommended)
            with torch.no_grad():
                random_img.data = torch.clamp(random_img.data, -2.5, 2.5)
    except Exception as e:
        print(f"Error in optimization: {e}")
    finally:
        handle.remove()
        
    return denormalize_tensor(random_img.squeeze(0).detach())

# =========================================================
# 2. FEATURE ATTRIBUTION (REPLACEMENT FOR OVERLAY)
# =========================================================

class GuidedBackprop:
    """
    Wrapper để thực hiện Guided Backpropagation (hoặc Vanilla Gradient nếu model không dùng ReLU).
    Dùng để visualize pixel nào trên ảnh đóng góp vào activation của Atom.
    """
    def __init__(self, model):
        self.model = model
        self.hooks = []
        self.model.eval()
        self.update_relus()

    def update_relus(self):
        """Register hook cho ReLU để lọc negative gradients (Guided Backprop logic)."""
        def relu_backward_hook_function(module, grad_in, grad_out):
            # Guided Backprop: Chỉ truyền gradient dương
            if isinstance(module, nn.ReLU):
                return (torch.clamp(grad_in[0], min=0.0),)
        
        for module in self.model.modules():
            if isinstance(module, nn.ReLU):
                self.hooks.append(module.register_backward_hook(relu_backward_hook_function))

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()

def compute_atom_saliency(model, atom, image_path, layer_idx, device):
    """
    Tính toán Saliency Map (Gradient-based) thể hiện vùng ảnh kích hoạt Atom.
    Đây là kỹ thuật tương tự DeConvolution/Guided Backprop.
    """
    # 1. Preprocess
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    try:
        img_pil = Image.open(image_path).convert('RGB')
        img_tensor = transform(img_pil).unsqueeze(0).to(device)
        img_tensor.requires_grad = True # Quan trọng: bật gradient cho input image
        
        # 2. Hook để lấy activation
        target_activations = []
        def hook_fn(module, input, output):
            target_activations.append(output)
        
        handle = model.feature_extractor[layer_idx].register_forward_hook(hook_fn)
        
        # 3. Init Guided Backprop (Optional: có thể bỏ qua nếu chỉ muốn Vanilla Gradient)
        # gbp = GuidedBackprop(model) 
        
        # 4. Forward
        model.zero_grad()
        _ = model(img_tensor)
        handle.remove()
        
        # 5. Calculate Loss based on Atom alignment
        act = target_activations[0].squeeze(0) # [C, H, W]
        spatial_activation = torch.mean(act, dim=0) # [H, W]
        atom_tensor = torch.tensor(atom, device=device).float()
        
        # Score = Dot product giữa Spatial Activation thực tế và Atom pattern
        # Chúng ta muốn tìm xem pixel nào làm cho Score này cao nhất
        score = torch.sum(spatial_activation * atom_tensor)
        
        # 6. Backward
        score.backward()
        
        # 7. Get Gradients at Input
        gradients = img_tensor.grad.data.cpu().numpy()[0] # [3, 224, 224]
        
        # gbp.remove_hooks() # Remove hook nếu dùng GuidedBackprop
        
        # 8. Convert to Grayscale Visualization (Saliency Map)
        # Lấy max absolute gradient qua các kênh màu
        saliency = np.max(np.abs(gradients), axis=0)
        
        # Normalize về 0-1
        if saliency.max() > 0:
            saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min())
            
        return saliency, img_pil.resize((224, 224))
        
    except Exception as e:
        print(f"Error computing saliency for {image_path}: {e}")
        return np.zeros((224, 224)), Image.new('RGB', (224, 224))

# =========================================================
# 3. VISUALIZATION CONTROLLER
# =========================================================

def visualize_class_unique_concepts(atom_analysis, D, model, layer_idx, device, subset_paths):
    unique_atoms = atom_analysis['unique_atoms']
    class_names = atom_analysis['class_names']
    
    class_unique_atoms = {}
    for atom_idx, class_idx in unique_atoms.items():
        if class_idx not in class_unique_atoms:
            class_unique_atoms[class_idx] = []
        class_unique_atoms[class_idx].append(atom_idx)
    
    classes_with_unique = sorted(class_unique_atoms.keys())
    n_classes = len(classes_with_unique)
    
    print(f"\nCreating visualizations for {n_classes} classes (Feature Reconstruction)...")
    
    n_cols = 3 
    n_rows = n_classes
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4*n_rows))
    if n_rows == 1: axes = axes.reshape(1, -1)
    
    for row_idx, class_idx in enumerate(classes_with_unique):
        atom_idx = class_unique_atoms[class_idx][0]
        
        if torch.is_tensor(D): atom_data = D[atom_idx].cpu().numpy()
        else: atom_data = D[atom_idx]
            
        # 1. Visual Pattern (Deep Dream / Optimization)
        opt_img = visualize_atom_input_space(model, atom_data, layer_idx, device=device)
        
        # 2. Saliency / Deconv (Gradient on Sample)
        sample_img_path = None
        for path, label in subset_paths:
            if label == class_idx:
                sample_img_path = path
                break
        
        if sample_img_path:
            saliency_map, orig_img = compute_atom_saliency(model, atom_data, sample_img_path, layer_idx, device)
        else:
            saliency_map, orig_img = np.zeros((224, 224)), None
        
        # --- Plotting ---
        
        # Col 1: Atom Abstract Pattern
        ax_atom = axes[row_idx, 0]
        im = ax_atom.imshow(atom_data, cmap='viridis')
        ax_atom.set_title(f"Class: {class_names[class_idx]}\nAtom {atom_idx}", fontsize=10, fontweight='bold')
        ax_atom.axis('off')
        
        # Col 2: Optimized Input (What the model 'dreams')
        ax_vis = axes[row_idx, 1]
        ax_vis.imshow(opt_img)
        ax_vis.set_title(f"Optimized Input\n(Model Dream)", fontsize=10)
        ax_vis.axis('off')
        
        # Col 3: Saliency Map (Where it looks in real image)
        ax_sal = axes[row_idx, 2]
        if orig_img:
            # Hiển thị Saliency map
            ax_sal.imshow(saliency_map, cmap='gray', vmin=0, vmax=1)
            # Optional: Blend nhẹ với ảnh gốc để dễ nhìn context (bỏ comment nếu muốn)
            # ax_sal.imshow(np.array(orig_img), alpha=0.3) 
        else:
            ax_sal.imshow(np.zeros((224, 224)), cmap='gray')
            
        ax_sal.set_title(f"Feature Attribution\n(Saliency/Deconv)", fontsize=10)
        ax_sal.axis('off')

    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/class_unique_concepts_saliency.png', dpi=150, bbox_inches='tight')
    print("\nSaved visualization to: figures/class_unique_concepts_saliency.png")
    plt.show()

# =========================================================
# RUN
# =========================================================
TARGET_LAYER_INDEX = 5
visualize_class_unique_concepts(atom_analysis, D, model, TARGET_LAYER_INDEX, device, subset_paths)